In [0]:
select * from adb_caresync_project.bronze.hospitals;

In [0]:
select * from adb_caresync_project.silver.hospitals;

In [0]:
select * from adb_caresync_project.bronze.doctors;

In [0]:
select * from adb_caresync_project.silver.doctors;

In [0]:
select * from adb_caresync_project.bronze.patients;

In [0]:
select * from adb_caresync_project.silver.patients;

In [0]:
select * from adb_caresync_project.bronze.insurance_providers;

In [0]:
%sql
-- Row count must equal silver.patients; patient_id must be unique; lab totals must reconcile with silver.lab_results
SELECT
  (SELECT COUNT(*) FROM adb_caresync_project.gold.patient_360)                     AS gold_rows,
  (SELECT COUNT(*) FROM adb_caresync_project.silver.patients)                      AS silver_patient_rows,
  (SELECT COUNT(DISTINCT patient_id) FROM adb_caresync_project.gold.patient_360)   AS distinct_patient_ids,
  (SELECT SUM(total_tests) FROM adb_caresync_project.gold.patient_360)             AS gold_total_tests,
  (SELECT COUNT(*) FROM adb_caresync_project.silver.lab_results
     WHERE patient_id IN (SELECT patient_id FROM adb_caresync_project.silver.patients)) AS silver_lab_rows_matched,
  (SELECT COUNT_IF(hospital_name IS NULL) FROM adb_caresync_project.gold.patient_360)          AS null_hospital_name,
  (SELECT COUNT_IF(insurance_provider_name IS NULL) FROM adb_caresync_project.gold.patient_360) AS null_insurance_name,
  (SELECT COUNT_IF(total_tests = 0) FROM adb_caresync_project.gold.patient_360)    AS patients_without_tests;

In [0]:
%sql
-- Row count should equal distinct (hospital_id, test_date) combos in lab_results
-- Test totals should reconcile with silver.lab_results
SELECT
  (SELECT COUNT(*) FROM adb_caresync_project.gold.hospital_daily_summary)                         AS gold_rows,
  (SELECT COUNT(DISTINCT CONCAT(hospital_id, '-', test_date))
     FROM adb_caresync_project.silver.lab_results)                                                AS expected_rows,
  (SELECT SUM(total_tests) FROM adb_caresync_project.gold.hospital_daily_summary)                 AS gold_total_tests,
  (SELECT COUNT(*) FROM adb_caresync_project.silver.lab_results)                                  AS silver_lab_rows,
  (SELECT COUNT_IF(hospital_name IS NULL) FROM adb_caresync_project.gold.hospital_daily_summary)  AS null_hospital_name,
  (SELECT COUNT_IF(day_name IS NULL) FROM adb_caresync_project.gold.hospital_daily_summary)       AS null_day_name,
  (SELECT COUNT_IF(prev_day_total_tests IS NULL) FROM adb_caresync_project.gold.hospital_daily_summary) AS null_prev_day;

In [0]:
%sql
-- Row count should equal distinct (test_name, year, month) combos in lab_results
-- Test totals should reconcile with silver.lab_results
SELECT
  (SELECT COUNT(*) FROM adb_caresync_project.gold.lab_test_trends)                                AS gold_rows,
  (SELECT COUNT(DISTINCT CONCAT(test_name, '-', YEAR(test_date), '-', MONTH(test_date)))
     FROM adb_caresync_project.silver.lab_results)                                                AS expected_rows,
  (SELECT SUM(total_tests) FROM adb_caresync_project.gold.lab_test_trends)                        AS gold_total_tests,
  (SELECT COUNT(*) FROM adb_caresync_project.silver.lab_results)                                  AS silver_lab_rows,
  (SELECT COUNT_IF(month_name IS NULL) FROM adb_caresync_project.gold.lab_test_trends)            AS null_month_name,
  (SELECT COUNT_IF(quarter_name IS NULL) FROM adb_caresync_project.gold.lab_test_trends)          AS null_quarter_name,
  (SELECT COUNT_IF(prev_month_abnormal_rate_pct IS NULL) FROM adb_caresync_project.gold.lab_test_trends) AS null_prev_month;